<a href="https://colab.research.google.com/github/Bishant321/Bishant321/blob/main/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.transforms.v2 as T

In [ ]:
# Define basic transformations using torchvision.transforms.v2
transforms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.5], std=[0.5])
])

# Creating a dummy dataset for demonstration purposes
class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, size=100, transform=None):
        self.size = size
        # Create 100 dummy 'images' (1 channel, 28x28) and random targets
        self.data = torch.randint(0, 256, (size, 1, 28, 28), dtype=torch.uint8)
        self.targets = torch.randint(0, 10, (size,))
        self.transform = transform

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.targets[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

dataset = DummyDataset(size=120, transform=transforms)

In [ ]:
# Split dataset into training (100 samples) and validation (20 samples)
train_indices = list(range(0, 100))
val_indices = list(range(100, 120))

train_set = Subset(dataset, train_indices)
val_set = Subset(dataset, val_indices)

# Configure hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 2

# Create DataLoaders
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# Verify configuration by fetching a single batch
images, labels = next(iter(train_loader))
print(f"Batch of images shape: {images.shape}")
print(f"Batch of labels shape: {labels.shape}")

Batch of images shape: torch.Size([16, 1, 28, 28])
Batch of labels shape: torch.Size([16])


In [ ]:
from torchvision.datasets import OxfordIIITPet

resize = T.Compose([
    T.Resize((64,64)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True)
])

transformed_data = OxfordIIITPet(root="data/", split="trainval", transform=resize, download=True)
train_loader = DataLoader(
    Subset(transformed_data, range(400)), batch_size=16, shuffle=True
)

In [ ]:
class simpleCNN(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.network = nn.Linear(32 * 16 * 16, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # 64 -> 32
        x = self.pool(torch.relu(self.conv2(x)))  # 32 -> 16
        x = torch.flatten(x, start_dim=1)
        return self.network(x)

model = simpleCNN()